# Cold-Start Popularity + Genre Coverage Demo

This notebook demonstrates a cold-start policy for `SVDModel` and `ItemKNNModel`:
we recommend only globally popular items and rerank them to cover a wider set of genres.

The goal is to avoid narrow recommendation lists for users with no personal history.
You can use this workflow as a practical fallback when personalization is not yet possible.

In [1]:
from pathlib import Path

import pandas as pd

from src.models.cold_start import BayesianColdStartRanker
from src.models.inference_router import RecommenderInferenceRouter
from src.models.item_knn_model import ItemKNNModel
from src.models.svd_model import SVDModel

## Load Data

We load cleaned ratings and movies metadata from the processed folder.
These two sources are enough to estimate popularity and genre coverage for fallback recommendations.

In [2]:
project_root_path = Path.cwd()
if not (project_root_path / "src").exists():
    project_root_path = project_root_path.parent

ratings_path = project_root_path / "data" / "processed" / "ratings_train_cleaned.csv"
movies_path = project_root_path / "data" / "processed" / "movies_cleaned.csv"

ratings_dataframe = pd.read_csv(ratings_path)
movies_dataframe = pd.read_csv(movies_path)

print(f"Ratings rows: {len(ratings_dataframe)}")
print(f"Movies rows: {len(movies_dataframe)}")

Ratings rows: 97801
Movies rows: 9742


## Fit Shared Cold-Start Ranker

The ranker stores popularity statistics and genre metadata used by the fallback strategy.
Keeping one shared ranker also makes behavior consistent across different underlying recommender models.

In [3]:
cold_start_ranker = BayesianColdStartRanker()
cold_start_ranker.fit(
    ratings_dataframe=ratings_dataframe,
    movies_dataframe=movies_dataframe,
)

## Route Cold-Start for SVD and ItemKNN

For unknown users, the router now uses `popular_genre_coverage` for Surprise models.
This lets both models return recommendations from the same fallback policy when user history is missing.

In [4]:
svd_router = RecommenderInferenceRouter(
    trained_model=SVDModel(),
    cold_start_ranker=cold_start_ranker,
    ratings_dataframe=ratings_dataframe,
    minimum_personalization_interactions=2,
)

itemknn_router = RecommenderInferenceRouter(
    trained_model=ItemKNNModel(),
    cold_start_ranker=cold_start_ranker,
    ratings_dataframe=ratings_dataframe,
    minimum_personalization_interactions=2,
)

cold_user_identifier = 999999
svd_result = svd_router.recommend_for_user(user_identifier=cold_user_identifier, number_of_recommendations=10)
itemknn_result = itemknn_router.recommend_for_user(user_identifier=cold_user_identifier, number_of_recommendations=10)

print("SVD source:", svd_result.source_name)
print("ItemKNN source:", itemknn_result.source_name)

SVD source: cold_start_fallback:popular_genre_coverage
ItemKNN source: cold_start_fallback:popular_genre_coverage


## Inspect Recommendations and Genre Spread

We show the recommended movies and count distinct genres in the list.
A higher number of unique genres usually means better topical coverage for cold-start users.

In [5]:
movie_title_map = (
    {
        int(movie_identifier): str(title_value)
        for movie_identifier, title_value in movies_dataframe[["movieId", "title"]].itertuples(index=False, name=None)
    }
    if "title" in movies_dataframe.columns
    else {}
)

movie_genre_map = {
    int(movie_identifier): str(genres_value)
    for movie_identifier, genres_value in movies_dataframe[["movieId", "genres"]].itertuples(index=False, name=None)
}


def summarize_result(result_name: str, recommendation_result) -> pd.DataFrame:
    summary_rows = []
    covered_genres = set()
    for rank_value, (movie_identifier, score_value) in enumerate(recommendation_result.recommendations, start=1):
        genre_text = movie_genre_map.get(int(movie_identifier), "")
        genre_tokens = {token.strip() for token in genre_text.split("|") if token.strip()}
        covered_genres.update(genre_tokens)
        summary_rows.append(
            {
                "model": result_name,
                "rank": rank_value,
                "movieId": int(movie_identifier),
                "title": movie_title_map.get(int(movie_identifier), "(title unavailable)"),
                "genres": genre_text,
                "score": float(score_value),
            }
        )
    summary_dataframe = pd.DataFrame(summary_rows)
    print(f"{result_name}: unique genres in top-10 = {len(covered_genres)}")
    return summary_dataframe


svd_summary_dataframe = summarize_result("svd", svd_result)
itemknn_summary_dataframe = summarize_result("itemknn", itemknn_result)

pd.concat([svd_summary_dataframe, itemknn_summary_dataframe], ignore_index=True)

,model,rank,movieId,title,genres,score
0,svd,1,356,Forrest Gump (1994),Comedy|Drama|Romance|War,326.0
1,svd,2,318,"Shawshank Redemption, The (1994)",Crime|Drama,309.0
2,svd,3,296,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller,303.0
3,svd,4,593,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller,276.0
4,svd,5,2571,"Matrix, The (1999)",Action|Sci-Fi|Thriller,271.0
5,svd,6,260,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi,244.0
6,svd,7,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,209.0
7,svd,8,50,"Usual Suspects, The (1995)",Crime|Mystery|Thriller,199.0
8,svd,9,150,Apollo 13 (1995),Adventure|Drama|IMAX,195.0
9,svd,10,588,Aladdin (1992),Adventure|Animation|Children|Comedy|Musical,177.0
